Pytest Unit Test Generator
Generate pytest unit tests from Python code using OpenAI or Ollama. Run tests and coverage in the app, edit generated tests if needed, and save them to the tests/ folder.

Models: gpt-4o-mini (OpenAI) and llama3.2 (Ollama). No other providers.

In [ ]:
# Install: pytest, pytest-cov (and pip install python-dotenv openai gradio for the app)
# !pip install pytest pytest-cov python-dotenv openai gradio

Flow
Paste your Python code (e.g. a function) in the left panel and pick OpenAI or Ollama.
Click Generate tests — the model streams pytest code into the right panel. You can edit it there (Ollama output often needs small fixes).
Click Run tests to execute pytest; Coverage runs coverage run -m pytest and shows the report.
Click Save tests to write the generated test file into tests/ (and remove the temporary test_code.py).

In [ ]:
import re
import os
import sys
import textwrap
from pathlib import Path
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

: 

In [ ]:
# Config: OpenAI and Ollama only
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")
if OPENAI_API_KEY:
    print(f"OpenAI API key loaded ({OPENAI_API_KEY[:4]}...)")
else:
    print("OPENAI_API_KEY not set — OpenAI model will fail.")

OPENAI_MODEL = "gpt-4o-mini"
OLLAMA_MODEL = "llama3.2:1b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openai_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
def extract_code(text):
    """Take code from markdown fenced block ```python ... ``` or return text as-is."""
    if not (text or text.strip()):
        return ""
    match = re.search(r"```python\s*(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def execute_coverage_report(python_interpreter=sys.executable):
    if not python_interpreter:
        raise EnvironmentError("Python interpreter not found.")
    try:
        result = subprocess.run(
            [python_interpreter, "-m", "coverage", "run", "-m", "pytest", "tests"],
            capture_output=True, text=True, cwd=os.getcwd(),
        )
        return result.stdout or result.stderr or "(no output)"
    except Exception as e:
        return str(e)


def save_unit_tests(code):
    """Save generated test into tests/test_<function_name>.py and remove tests/test_code.py."""
    raw = extract_code(code)
    match = re.search(r"def\s+test_(\w+)\s*\(", raw) or re.search(r"def\s+(\w+)\s*\(", code)
    name = match.group(1).strip() if match else "generated"
    tests_dir = Path("tests")
    tests_dir.mkdir(parents=True, exist_ok=True)
    (tests_dir / f"test_{name}.py").write_text(raw, encoding="utf-8")
    tmp = tests_dir / "test_code.py"
    if tmp.exists():
        tmp.unlink()
    return f"Saved tests/test_{name}.py"


def execute_tests_in_venv(code_to_test, tests, python_interpreter=sys.executable):
    """Run pytest with source + generated test in tests/test_code.py."""
    if not python_interpreter:
        raise EnvironmentError("Python interpreter not found.")
    combined = textwrap.dedent(code_to_test or "") + "\n\n" + extract_code(tests or "")
    tests_dir = Path("tests")
    tests_dir.mkdir(parents=True, exist_ok=True)
    (tests_dir / "test_code.py").write_text(combined, encoding="utf-8")
    try:
        result = subprocess.run(
            [python_interpreter, "-m", "pytest", str(tests_dir), "-v"],
            capture_output=True, text=True, cwd=os.getcwd(),
        )
        return result.stdout or result.stderr or "(no output)"
    except Exception as e:
        return str(e)

In [ ]:
SYSTEM_PROMPT = """You are a pytest expert. Generate unit tests for the user's Python code. Output only the test module: no preamble, no explanation. Use pytest; include imports, pytest.raises for expected exceptions, and clear test names (e.g. test_<function>_normal_case, test_<function>_raises_type_error). Wrap the code in a markdown fenced block: ```python ... ```"""

In [ ]:
def build_user_prompt(code):
    """Build the user message that asks the model for pytest code."""
    return """Generate pytest unit tests for the following Python code. Cover:
- Happy path and typical inputs.
- Edge cases and boundary conditions.
- Invalid inputs or cases that must raise (use pytest.raises).
Use only standard library or pytest; do not include the code under test in the output. Reply with a single ```python ... ``` block.

Code to test:

""" + (code or "")

In [ ]:
def stream_openai(code):
    """Stream pytest code from gpt-4o-mini."""
    user_prompt = build_user_prompt(code)
    stream = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


def stream_ollama(code):
    """Stream pytest code from Ollama (llama3.2)."""
    user_prompt = build_user_prompt(code)
    stream = ollama_client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

Code examples to test the inteface

In [ ]:
function_to_test = """
    def lengthOfLongestSubstring(s):
        if not isinstance(s, str):
            raise TypeError("Input must be a string")
        max_length = 0
        substring = ""
        start_idx = 0
        while start_idx < len(s):
            string = s[start_idx:]
            for i, x in enumerate(string):
                substring += x
                if len(substring) == len(set((list(substring)))):
                    
                    if len(set((list(substring)))) > max_length:
                        
                        max_length = len(substring)

            start_idx += 1
            substring = ""
                  
                
        return max_length"""

In [ ]:
test_code = """```python
import pytest

# Unit tests using pytest
def test_lengthOfLongestSubstring():
    assert lengthOfLongestSubstring("abcabcbb") == 3  # Case with repeating characters
    assert lengthOfLongestSubstring("bbbbb") == 1    # Case with all same characters
    assert lengthOfLongestSubstring("pwwkew") == 3    # Case with mixed characters
    assert lengthOfLongestSubstring("") == 0           # Empty string case
    assert lengthOfLongestSubstring("abcdef") == 6     # All unique characters
    assert lengthOfLongestSubstring("abca") == 3       # Case with pattern and repeat
    assert lengthOfLongestSubstring("dvdf") == 3       # Case with repeated characters separated
    assert lengthOfLongestSubstring("a") == 1           # Case with single character
    assert lengthOfLongestSubstring("au") == 2          # Case with unique two characters
```"""

In [ ]:
def generate_tests(code, model):
    """Stream generated pytest code; model is 'OpenAI' or 'Ollama'."""
    if model == "OpenAI":
        gen = stream_openai(code)
    elif model == "Ollama":
        gen = stream_ollama(code)
    else:
        raise ValueError("Model must be OpenAI or Ollama")
    for chunk in gen:
        yield chunk

Gradio interface

In [ ]:

with gr.Blocks(title="Pytest Unit Test Generator", theme=gr.themes.Soft()) as ui:
    gr.Markdown("## Pytest unit test generator (OpenAI or Ollama)")
    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            python = gr.Textbox(label="Python code to test", value=function_to_test, lines=14)
            model = gr.Dropdown(["OpenAI", "Ollama"], label="Model", value="OpenAI")
            unit_tests_btn = gr.Button("Generate tests", variant="primary")
        with gr.Column(scale=1, min_width=280):
            unit_tests_out = gr.Textbox(label="Generated pytest code", value=test_code, lines=14)
            run_btn = gr.Button("Run tests")
            coverage_btn = gr.Button("Coverage report")
            save_btn = gr.Button("Save tests")
    with gr.Row():
        python_out = gr.Textbox(label="Test run output", lines=8)
        coverage_out = gr.Textbox(label="Coverage report", lines=8)

    unit_tests_btn.click(generate_tests, inputs=[python, model], outputs=unit_tests_out)
    run_btn.click(execute_tests_in_venv, inputs=[python, unit_tests_out], outputs=python_out)
    coverage_btn.click(execute_coverage_report, outputs=coverage_out)
    save_btn.click(save_unit_tests, inputs=unit_tests_out, outputs=python_out)

ui.launch()